In [1]:
# Testing Gridsynth
import random
from pygridsynth.gridsynth import gridsynth_gates
from bqskit.ir import Circuit
from bqskit.qis import UnitaryMatrix
from bqskit.ir.gates import *
from util import normalized_gp_frob_cost, gp_frobenius_cost, fix_phase

gate_defs = {'I': IdentityGate(1), 'Z': ZGate(), 'S': SGate(), 'Sd': SdgGate(), 
             'T': TGate(), 'Td': TdgGate(), 'H': HGate(), 'D': SdgGate(), 
             'X': XGate(), 'L': SdgGate(), 'Y': YGate()}

def gridsynth_gates_to_cir(gates: str):
    circ = Circuit(1)
    # Loop through string and add gates to circuit
    while len(gates) > 0:
        # Check 2-character gates first
        next_token = gates[:2]
        if next_token in gate_defs:
            circ.append_gate(gate_defs[next_token], (0,))
            gates = gates[2:]
        else:
            # Otherwise, check 1-character gates
            next_token = gates[0]
            if next_token in gate_defs:
                circ.append_gate(gate_defs[next_token], (0,))
            # else:
                # print("Unknown: ", next_token)
            gates = gates[1:]
    # Return the circuit
    return circ


def remove_t(s):
    indices = [i for i, c in enumerate(s) if c == 'T']
    if not indices:
        return s  # No 'T' to remove

    remove_idx = random.choice(indices)
    return s[:remove_idx] + s[remove_idx + 1:]

def delete_all_possible_ts(angle: float, epsilon: float):
    orig_str = gridsynth_gates(angle, epsilon ** 2)
    num_fails = 0
    num_ts_removed = 0
    rz = RZGate()
    target = rz.get_unitary([angle])
    orig_grid_circ = gridsynth_gates_to_cir(orig_str)
    orig_cost = normalized_gp_frob_cost(target, orig_grid_circ.get_unitary())
    assert orig_cost < epsilon
    f = orig_str
    while num_fails < 50:                
        new_grid_str = remove_t(f)
        new_grid_circ = gridsynth_gates_to_cir(new_grid_str)
        new_cost = normalized_gp_frob_cost(target, new_grid_circ.get_unitary())
        if new_cost < epsilon:
            f = new_grid_str
            num_ts_removed += 1
        else:
            num_fails += 1 
    return num_ts_removed

In [2]:
from mpmath import mp
import numpy as np

mp.dps = 20
eps = 1e-4
angle = np.random.uniform(0, 2 * np.pi)
V = RZGate().get_unitary([angle])
t_str = gridsynth_gates(angle, eps)
t_circ = gridsynth_gates_to_cir(t_str)
orig_unitary = t_circ.get_unitary()
orig_t_count = t_str.count("T")
new_angle = RZGate.calc_params(t_circ.get_unitary())
new_t_str = gridsynth_gates(new_angle, eps)
new_t_count = new_t_str.count("T")
new_unitary = gridsynth_gates_to_cir(new_t_str).get_unitary()
print(orig_t_count, new_t_count)
print(f"Cost: {gp_frobenius_cost(orig_unitary, new_unitary)}")
print(f"Cost: {gp_frobenius_cost(V, new_unitary)}")
print(f"Cost: {gp_frobenius_cost(orig_unitary, V)}")

38 38
Cost: 0.0
Cost: 4.121990908788677e-05
Cost: 4.121990908788677e-05


In [3]:
from util import gp_frobenius_cost
from bqskit.utils.math import unitary_log_no_i, pauli_expansion
import numpy as np

# Now test Pauli Twirling with gridsynth
def t_count(circ: Circuit):
    return circ.count(TGate()) + circ.count(TdgGate())

np.set_printoptions(precision=5, threshold=np.inf, linewidth=np.inf)

# twirl_gates = [(ZGate(), ZGate()), (SGate(), SdgGate()), (SdgGate(), SGate())]

# Just do Z Twirling
twirl_gates = [(ZGate(), ZGate())]

# Perturb angle by a small amount
epsilones = [1e-2, 1e-3, 1e-5]
for epsilon in epsilones:
    starting_angle = np.random.uniform(-1 * np.pi, 1 * np.pi)
    V = RZGate().get_unitary([starting_angle])
    full_t_str = gridsynth_gates(starting_angle, epsilon ** 2)
    full_t_circ = gridsynth_gates_to_cir(full_t_str)
    orig_dist = gp_frobenius_cost(full_t_circ.get_unitary(), V)
    orig_t_count = t_count(full_t_circ)

    U_1_t_str = gridsynth_gates(starting_angle, epsilon)
    U_1_t_circ = gridsynth_gates_to_cir(U_1_t_str)
    U_1 = U_1_t_circ.get_unitary()
    U_1_ang = RZGate.calc_params(U_1)

    Vt_U_1 = V.conj().T @ U_1

    # Expand to Pauli Basis and get Z component
    # az = np.imag(Vt_U_1[0, 0])
    H = unitary_log_no_i(Vt_U_1)
    coeffs = pauli_expansion(H)
    az = coeffs[-1]

    delta = 2 * np.arcsin(np.sqrt(epsilon) / 2)
    if az < 0:
        delta = -delta
    
    V_p = RZGate().get_unitary([starting_angle + delta])
    U_2_t_str = gridsynth_gates(starting_angle + delta, epsilon )
    U_2_t_circ = gridsynth_gates_to_cir(U_2_t_str)
    U_2 = U_2_t_circ.get_unitary()

    v_v_p_dist = gp_frobenius_cost(V, V_p)
    # eps = v_v_p_dist
    u2_v_dist = gp_frobenius_cost(U_2_t_circ.get_unitary(), V)
    u1_vp_dist = gp_frobenius_cost(U_1_t_circ.get_unitary(), V_p)
    perturbed_t_circs = [U_1_t_circ, U_2_t_circ]

    for gate, gate_dg in twirl_gates:
        # Twirl U_1
        new_circ_1 = U_1_t_circ.copy()
        new_circ_1.append_gate(gate, (0,))
        new_circ_1.insert_gate(0, gate_dg, (0,))
        # Now do with U_2
        new_circ_2 = U_2_t_circ.copy()
        new_circ_2.append_gate(gate, (0,))
        new_circ_2.insert_gate(0, gate_dg, (0,))
        
        perturbed_t_circs.append(new_circ_1)
        perturbed_t_circs.append(new_circ_2)

    print("Num Perturbed Circuits: ", len(perturbed_t_circs))

    [fix_phase(c, V) for c in perturbed_t_circs]
    perturbed_unitaries = [c.get_unitary() for c in perturbed_t_circs]
    perturbed_t_counts = [t_count(c) for c in perturbed_t_circs]
    perturbed_costs = [gp_frobenius_cost(un, V) for un in perturbed_unitaries]

    Vt_U_2 = V.conj().T @ U_2_t_circ.get_unitary()
    H = unitary_log_no_i(Vt_U_2)
    coeffs = pauli_expansion(H)
    Bz = coeffs[-1]

    q = az / (az - Bz)

    p2 = q / 2
    p1 = (1 - q) / 2

    probs = [p1, p2] * 2

    mean_un = np.average(perturbed_unitaries, axis=0, weights=probs)

    cost_of_mean = gp_frobenius_cost(mean_un, V)
    eps = np.mean(perturbed_costs)
    ratio = cost_of_mean / eps / eps
    # print(perturbed_costs)
    print("Orig Distance: ", orig_dist, " Mean Unitary Distance: ", cost_of_mean, "Ratio: ", ratio)
    print("Orig T Count: ", orig_t_count, "Avg T Count: ", np.mean(perturbed_t_counts))

Num Perturbed Circuits:  4
Orig Distance:  0.00012072885662331879  Mean Unitary Distance:  5.577950647620573e-05 Ratio:  0.02927023802878429
Orig T Count:  40 Avg T Count:  17.0
Num Perturbed Circuits:  4
Orig Distance:  1.1460317955565298e-06  Mean Unitary Distance:  9.329729041647815e-07 Ratio:  0.007062441937240601
Orig T Count:  60 Avg T Count:  31.0
Num Perturbed Circuits:  4
Orig Distance:  4.873173018516283e-11  Mean Unitary Distance:  4.968431645693073e-09 Ratio:  0.003906544614279521
Orig T Count:  106 Avg T Count:  46.0


In [4]:
def get_default_rz_circ(starting_angle, epsilon):
    default_t_str = gridsynth_gates(starting_angle, epsilon ** 2)
    default_t_circ = gridsynth_gates_to_cir(default_t_str)
    return default_t_circ


# Turn into a function
def get_rz_perturbations(starting_angle, epsilon) -> tuple[list[Circuit], 
                                                       list[float]]:
    V = RZGate().get_unitary([starting_angle])
    U_1_t_str = gridsynth_gates(starting_angle, epsilon)
    U_1_t_circ = gridsynth_gates_to_cir(U_1_t_str)
    U_1 = U_1_t_circ.get_unitary()

    Vt_U_1 = V.conj().T @ U_1

    # Expand to Pauli Basis and get Z component
    H = unitary_log_no_i(Vt_U_1)
    coeffs = pauli_expansion(H)
    az = coeffs[-1]

    delta = 2 * np.arcsin(np.sqrt(epsilon) / 2)
    if az < 0:
        delta = -delta
    
    U_2_t_str = gridsynth_gates(starting_angle + delta, epsilon )
    U_2_t_circ = gridsynth_gates_to_cir(U_2_t_str)

    perturbed_t_circs = [U_1_t_circ, U_2_t_circ]

    for gate, gate_dg in twirl_gates:
        # Twirl U_1
        new_circ_1 = U_1_t_circ.copy()
        new_circ_1.append_gate(gate, (0,))
        new_circ_1.insert_gate(0, gate_dg, (0,))
        # Now do with U_2
        new_circ_2 = U_2_t_circ.copy()
        new_circ_2.append_gate(gate, (0,))
        new_circ_2.insert_gate(0, gate_dg, (0,))
        perturbed_t_circs.append(new_circ_1)
        perturbed_t_circs.append(new_circ_2)

    [fix_phase(c, V) for c in perturbed_t_circs]
    perturbed_unitaries = [c.get_unitary() for c in perturbed_t_circs]
    perturbed_t_counts = [t_count(c) for c in perturbed_t_circs]
    perturbed_costs = [gp_frobenius_cost(un, V) for un in perturbed_unitaries]

    Vt_U_2 = V.conj().T @ U_2_t_circ.get_unitary()
    H = unitary_log_no_i(Vt_U_2)
    coeffs = pauli_expansion(H)
    Bz = coeffs[-1]

    q = az / (az - Bz)

    p2 = q / 2
    p1 = (1 - q) / 2

    probs = [p1, p2] * 2

    print(probs)

    if p1 < 0 or p2 < 0:
        print("Negative probs: ", p1, p2)

    mean_un = np.average(perturbed_unitaries, axis=0, weights=probs)

    cost_of_mean = gp_frobenius_cost(mean_un, V)
    eps = np.mean(perturbed_costs)
    ratio = cost_of_mean / eps / eps
    if ratio > 5:
        print("Bad Ratio!")
        default = get_default_rz_circ(starting_angle, epsilon)
        return [default], [1]
    # print(perturbed_costs)
    # print(" Mean Unitary Distance: ", cost_of_mean, "Ratio: ", ratio)
    # print("Avg T Count: ", np.mean(perturbed_t_counts))
    return perturbed_t_circs, probs

In [5]:
starting_angle = np.random.uniform(-1 * np.pi, 1 * np.pi)
for epsilon in epsilones:
    print(epsilon)
    perturbed_t_circs, probs = get_rz_perturbations(starting_angle, epsilon)
    print(np.sum(probs))

0.01
[0.4854890468054981, 0.014510953194501897, 0.4854890468054981, 0.014510953194501897]
1.0
0.001
[0.49728705710250953, 0.002712942897490493, 0.49728705710250953, 0.002712942897490493]
1.0
1e-05
[0.4989336468621534, 0.0010663531378465605, 0.4989336468621534, 0.0010663531378465605]
1.0


In [6]:
# Create a random circuit with CNOTs and RZs
def rand_circ(num_qubits: int = 3, num_rzs: int = 3):
    circ = Circuit(num_qubits)
    for _ in range(num_rzs):
        qubit = np.random.randint(0, num_qubits)
        angle = np.random.uniform(-1 * np.pi, 1 * np.pi)
        circ.append_gate(RZGate(), (qubit,), [angle])
        if np.random.random() < 0.8 and num_qubits > 1:
            qubit_2 = np.random.choice([i for i in range(num_qubits) if i != qubit])
            circ.append_gate(CNOTGate(), (qubit, qubit_2))
        # Add additional H, S, Sdg, and X gates
        if np.random.random() < 0.5:
            circ.append_gate(XGate(), (qubit,))
        if np.random.random() < 0.5:
            circ.append_gate(HGate(), (qubit,))
        if np.random.random() < 0.5:
            circ.append_gate(SGate(), (qubit,))
        if np.random.random() < 0.5:
            circ.append_gate(SdgGate(), (qubit,))
    return circ

In [7]:
import itertools
from bqskit.ir.circuit import CircuitPoint
# For each angle in a circuit, get the perturbation circuits
# There will be 8 ^ (num RZs) circuits
def get_perturbation_circuits(circ: Circuit, epsilon: float) -> tuple[list[Circuit], list[float]]:
    num_rzs = circ.count(RZGate())
    rz_perturbations = [[]] * num_rzs
    rz_probs = [[]] * num_rzs
    i = 0   
    for cycle, op in circ.operations_with_cycles():
        if isinstance(op.gate, RZGate):
            angle = op.params[0]
            circs, probs = get_rz_perturbations(angle, epsilon)
            print(np.sum(probs))
            assert(np.allclose(np.sum(probs), 1))
            # print(np.sum(probs))
            rz_perturbations[i] = circs
            rz_probs[i] = probs
            i += 1
    rz_perturbation_inds = [list(range(len(p))) for p in rz_perturbations]
    perturbed_circs = []
    # for each perturbation, get the cartesian product of all the perturbations
    rz_combos = list(itertools.product(*rz_perturbation_inds))
    # assert len(rz_combos) == 8 ** num_rzs
    all_probs = []
    for rz_combo in rz_combos:
        # print(rz_combo)
        perturbed_circ = circ.copy()
        i = 0
        prob = 1
        for cycle, op in perturbed_circ.operations_with_cycles():
            if isinstance(op.gate, RZGate):
                perturbation_circ = rz_perturbations[i][rz_combo[i]]
                prob *= rz_probs[i][rz_combo[i]]
                # print(prob)
                perturbed_circ.replace_with_circuit(CircuitPoint(cycle, op.location[0]), perturbation_circ, as_circuit_gate=True)
                i += 1
            # perturbed_circ.insert_circuit(0, perturbation_circ, (i,))
            # perturbed_circ.set_params(probs)
        perturbed_circ.unfold_all()
        perturbed_circs.append(perturbed_circ)
        all_probs.append(prob)

    # print(np.sum(all_probs))
    assert(np.allclose(np.sum(all_probs), 1))
    return perturbed_circs, all_probs

In [8]:
circ = rand_circ(2, 2)
print(circ.gate_counts)
p_circs, probs = get_perturbation_circuits(circ, 1e-3)

{SdgGate: 1, CNOTGate: 2, RZGate: 2}
[0.006487902535496914, 0.4935120974645031, 0.006487902535496914, 0.4935120974645031]
Bad Ratio!
1
[0.4966391793389175, 0.0033608206610825048, 0.4966391793389175, 0.0033608206610825048]
1.0


In [9]:
def get_default_circ(circ: Circuit, epsilon: float) -> Circuit:
    # For each RZ, replace with default_rz_circ
    new_circ = circ.copy()
    for cycle, op in circ.operations_with_cycles():
        if isinstance(op.gate, RZGate):
            # Get default circ
            default_circ = get_default_rz_circ(op.params[0], epsilon)
            # Replace with default circ
            new_circ.replace_with_circuit(CircuitPoint(cycle, op.location[0]), default_circ, as_circuit_gate=True)
    new_circ.unfold_all()
    return new_circ

In [16]:
# p_circs, probs = p_circs
eps = 1e-2
num_rzs = 6
circ = rand_circ(3, num_rzs)
print(circ.gate_counts)
p_circs, probs = get_perturbation_circuits(circ, eps)

max_ens_size = len(p_circs)
print("Max Ense Size: ", max_ens_size)
print("Avg T Count: ", np.mean([t_count(c) for c in p_circs]))

# ens_sizes = np.geomspace(1, max_ens_size, num=8).astype(int)
ens_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 4096]

dists = []

target = circ.get_unitary()

default_circ = get_default_circ(circ, eps)
default_dist = normalized_gp_frob_cost(default_circ.get_unitary(), target)
print("Default T Count: ", t_count(default_circ))

for size in ens_sizes:
    # Randomly select from circs
    rand_inds = np.random.choice(len(p_circs), size=size, p=probs)
    rand_circs = [p_circs[i] for i in rand_inds]
    # Get the average unitary
    rand_unitaries = [c.get_unitary() for c in rand_circs]
    rand_unitaries = np.array(rand_unitaries)
    avg_unitary = np.average(rand_unitaries, axis=0)
    # Get the cost
    cost = normalized_gp_frob_cost(avg_unitary, target)
    dists.append(cost)

{HGate: 2, SdgGate: 1, CNOTGate: 3, XGate: 1, RZGate: 6, SGate: 4}
[0.49076708623265014, 0.009232913767349846, 0.49076708623265014, 0.009232913767349846]
0.9999999999999999
[0.01008797873699413, 0.48991202126300587, 0.01008797873699413, 0.48991202126300587]
Bad Ratio!
1
[0.48872581208556526, 0.011274187914434753, 0.48872581208556526, 0.011274187914434753]
1.0
[0.4258473097383103, 0.0741526902616897, 0.4258473097383103, 0.0741526902616897]
1.0
[0.4607853203572731, 0.039214679642726916, 0.4607853203572731, 0.039214679642726916]
1.0
[0.4743071149296946, 0.02569288507030543, 0.4743071149296946, 0.02569288507030543]
1.0
Max Ense Size:  1024
Avg T Count:  130.0
Default T Count:  236


In [28]:
import matplotlib.pyplot as plt
# Plot dists vs ensemble size
fig, ax = plt.subplots(1,1)
print(dists)
plt.plot(ens_sizes, dists)
ax.plot(ens_sizes, dists)
ax.hlines([eps], xmin= 0, xmax = 4096, linestyles='--', colors='black')
ax.hlines([default_dist], xmin= 0, xmax = 4096, linestyles='--', colors='red')
ax.set_xlabel('Ensemble Size')
ax.set_ylabel('Distance')
ax.set_yscale('log')
# fig.show()
fig.savefig("test.png")

[0.014568689149152277, 0.00884178702945866, 0.007950912840583182, 0.014429692841476535, 0.007094665231800473, 0.002430419621517823, 0.0027579007231123918, 0.0016814417189080614, 0.0019239822331823852, 0.0011702160345775895, 0.0005589915359148457, 0.00045071031052101277]


ValueError: object __array__ method not producing an array

Error in callback <function _draw_all_if_interactive at 0x7f5f0ffba9e0> (for post_execute):


ValueError: object __array__ method not producing an array

ValueError: object __array__ method not producing an array

<Figure size 640x480 with 1 Axes>

In [11]:
from util import GridSynthGate, normalized_gp_frob_cost, fix_phase
from bqskit.ir.gates import RZGate
import numpy as np

starting_angle = np.random.uniform(-1 * np.pi, 1 * np.pi)
epsilon = 7

params, probs = GridSynthGate.get_rz_perturbation_params(starting_angle, epsilon)
V = RZGate().get_unitary([starting_angle])
circs = [GridSynthGate().get_circuit(p) for p in params]
[fix_phase(c, V) for c in circs]
unitaries = [c.get_unitary() for c in circs]
costs = [normalized_gp_frob_cost(un, V) for un in unitaries]
mean_un = np.average(unitaries, axis=0, weights=probs)
print(normalized_gp_frob_cost(mean_un, V))
print(np.mean(costs))

1.7934594987381116e-08
5.592629035242157e-05


In [12]:
# from bqskit.ir.gates import *
# from util import normalized_gp_frob_cost
# import matplotlib.pyplot as plt
# from mpmath import mp

# mp.dps = 20
# # Plot epsilon vs. normalized frobenius cost
# min_eps = 1e-10
# max_eps = 1e-3

# epsilons = np.linspace(min_eps, max_eps, 100)

# def get_avg_frob_cost(epsilon: float) -> float:
#     final_costs = []
#     for _ in range(10):
#         starting_angle = np.random.uniform(-1 * np.pi, 1 * np.pi)
#         try:
#             params, probs = GridSynthGate.get_rz_perturbation_params(starting_angle, epsilon)
#         except:
#             continue
#         if len(params) == 1:
#             continue
#         V = RZGate().get_unitary([starting_angle])
#         uns = [GridSynthGate().get_unitary(p) for p in params]
#         costs = [normalized_gp_frob_cost(u, V) for u in uns]
#         final_costs.append(costs)
#     return np.mean(final_costs)

# frob_costs = [get_avg_frob_cost(e) for e in epsilons]
# fig, ax = plt.subplots()
# ax.plot(epsilons, frob_costs)
# ax.set_xlabel('Epsilon')
# ax.set_ylabel('Normalized Frobenius Cost')
# ax.set_yscale('log')
# ax.set_xscale('log')

# fig.show()


In [13]:
RZGate().get_unitary([np.pi])

array([[6.12323e-17-1.j, 0.00000e+00+0.j],
       [0.00000e+00+0.j, 6.12323e-17+1.j]])

In [14]:
# Time test
import math
from fractions import Fraction
import time
from pyLIQTR.gate_decomp.gate_approximation import approximate_rz_direct

def closest_rational_angle(angle, max_denominator=1000, in_degrees=False):
    """Find the closest rational approximation to an angle.

    Args:
        angle (float): The angle in radians (default) or degrees.
        max_denominator (int): Maximum denominator for the rational approximation.
        in_degrees (bool): If True, treats the input as degrees.

    Returns:
        Fraction: The closest rational fraction.
    """
    # if in_degrees:
    #     angle = math.radians(angle)  # Convert to radians

    # # Normalize angle to [0, 2π]
    # angle = angle % (2 * math.pi)

    # Approximate using fractions
    frac = Fraction(angle)

    return frac


def get_approx_t_str(angle, precision):
    if np.allclose(angle, 0):
        return "I", 0, 1
    if np.allclose(angle, 2):
        return "I", 2, 1
    num, den = closest_rational_angle(angle).as_integer_ratio()
    print("Angle: ", angle, "Num: ", num, "Den: ", den)
    return approximate_rz_direct(num, den, precision)[0], num, den



starting_angle = np.random.uniform(0, 2*np.pi)

start = time.process_time()
t_str, n, d = get_approx_t_str(starting_angle / np.pi, 2.5)

finish = time.process_time() - start

print(f"Time: {finish}")

# Convert to circuit
c = gridsynth_gates_to_cir(t_str)


# print(c.gate_counts)
# Get unitary
unitary = c.get_unitary()
# Get cost
target = RZGate().get_unitary([starting_angle])
target_2 = RZGate().get_unitary([n * np.pi / d])
cost = normalized_gp_frob_cost(unitary, target)
target_diff = normalized_gp_frob_cost(target, target_2)
print(f"Cost: {cost}", "Starting Angle: ", starting_angle)
print(f"Target Diff: {target_diff}")



Angle:  1.905260916653292 Num:  8580532354283335 Den:  4503599627370496


ValueError: invalid digits

In [ ]:


# Compare get_approx_t_str with gridsynth_gates
precisions = np.linspace(1, 10, 10).astype(int)
gridsynth_precisions = [10 ** -i for i in precisions]


gridsynth_start = time.time()


